# DBSCAN - multi-run experiments

In [ ]:
import sys
import time
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

sys.path.append("../src")
from data import make_optuna_subsample, make_final_subsample
from optuna_utils import run_study
from metrics import find_best_f1_threshold, minmax_scale_scores, evaluate_scores, print_metrics
from results import build_experiment_record, save_record_json, get_memory_mb

In [ ]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["CICIDS", "UNSW_NB15"]
SEED = 29
N_TRIALS = 200

DATASET_VERSION = "v1"
PREPROCESSING_VERSION = "v1"
SPLIT_METHOD = "stratified_train_val_test_fixed_seed"
MODEL_TYPE = "DBSCAN"
FUSION_STRATEGY = "none"

RUN_CONFIGS = [
    dict(run_index=1, n_train_opt=7000,  n_val_opt=3000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run1_small_opt_sample"),
    dict(run_index=2, n_train_opt=21000, n_val_opt=9000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run2_medium_opt_sample"),
    dict(run_index=3, n_train_opt=35000, n_val_opt=15000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run3_large_opt_sample"),
]

# ---------------------------------------------------------------------------
# NOTE on DBSCAN as an anomaly detector:
# DBSCAN has no native predict/score_samples for unseen points - it only
# assigns cluster labels to the data it was fit on. To score held-out
# val/test points we fit DBSCAN on the training data, keep the "core samples"
# it discovers as a reference set of "normal" density regions, and define the
# anomaly score of any new point as its distance to the nearest core sample
# (via a NearestNeighbors index). Larger distance -> more anomalous.
# If DBSCAN finds zero core samples for a given (eps, min_samples) pair, the
# trial is pruned since no meaningful scoring is possible.
# ---------------------------------------------------------------------------

In [ ]:
def _fit_reference_index(train_x, eps, min_samples, metric):
    db = DBSCAN(eps=eps, min_samples=min_samples, metric=metric, n_jobs=-1)
    labels = db.fit_predict(train_x)

    core_idx = db.core_sample_indices_
    if len(core_idx) == 0:
        return None, None

    core_points = np.asarray(train_x)[core_idx]
    nn = NearestNeighbors(n_neighbors=1, metric=metric, n_jobs=-1)
    nn.fit(core_points)
    return db, nn


def _score_with_reference(nn, x):
    distances, _ = nn.kneighbors(x, n_neighbors=1)
    return distances.ravel()


def make_objective(train_x, train_y, val_x, val_y):
    def objective(trial):
        n_pos = int(val_y.sum())
        n_neg = len(val_y) - n_pos
        if n_pos < 5 or n_neg < 5:
            raise optuna.exceptions.TrialPruned()

        eps = trial.suggest_float("eps", 0.05, 5.0, log=True)
        min_samples = trial.suggest_int("min_samples", 3, 50, log=True)
        metric = trial.suggest_categorical("metric", ["euclidean", "manhattan"])

        db, nn = _fit_reference_index(train_x, eps, min_samples, metric)
        if nn is None:
            raise optuna.exceptions.TrialPruned()

        try:
            val_scores = _score_with_reference(nn, val_x)
            auc = roc_auc_score(val_y, val_scores)
        except ValueError:
            raise optuna.exceptions.TrialPruned()

        return auc
    return objective


def fit_and_score_dbscan(params, train_x, train_y, val_x, test_x):
    gc.collect()
    mem_before = get_memory_mb()

    start_train = time.time()
    db, nn = _fit_reference_index(train_x, params["eps"], params["min_samples"], params["metric"])
    runtime_train = time.time() - start_train
    mem_after_train = get_memory_mb()

    if nn is None:
        raise RuntimeError(
            "Final DBSCAN fit produced zero core samples for the chosen "
            "hyperparameters; cannot score val/test points."
        )

    val_scores_raw = _score_with_reference(nn, val_x)
    scores_val = minmax_scale_scores(val_scores_raw)

    start_inference = time.time()
    test_scores_raw = _score_with_reference(nn, test_x)
    runtime_inference = time.time() - start_inference
    mem_after_inference = get_memory_mb()

    scores_test = minmax_scale_scores(test_scores_raw)
    memory_peak = max(mem_before, mem_after_train, mem_after_inference)

    return db, nn, scores_val, scores_test, runtime_train, runtime_inference, memory_peak

In [ ]:
def run_experiment(dataset, run_cfg):
    run_index = run_cfg["run_index"]
    study_name = f"DBSCAN_{dataset}_run{run_index}"

    train_x, train_y, val_x, val_y = make_optuna_subsample(
        dataset, SEED, run_cfg["n_train_opt"], run_cfg["n_val_opt"]
    )
    objective = make_objective(train_x, train_y, val_x, val_y)
    study = run_study(objective, study_name, SEED, N_TRIALS, results_dir=RESULTS_DIR)

    train_x, train_y, val_x, val_y, test_x, test_y = make_final_subsample(
        dataset, SEED,
        run_cfg["n_train_final"], run_cfg["n_val_final"], run_cfg["n_test_final"],
    )

    db, nn, scores_val, scores_test, runtime_train, runtime_inference, memory_peak = fit_and_score_dbscan(
        study.best_params, train_x, train_y, val_x, test_x
    )

    best_threshold, best_f1_val, best_prec_val, best_rec_val = find_best_f1_threshold(val_y, scores_val)
    print(f"[{dataset} run{run_index}] threshold={best_threshold:.4f} "
          f"F1={best_f1_val:.4f} P={best_prec_val:.4f} R={best_rec_val:.4f}")

    metrics = evaluate_scores(test_y, scores_test, threshold=best_threshold)
    print_metrics(f"DBSCAN final - {dataset} run{run_index}", metrics)

    model_dir = MODELS_DIR / MODEL_TYPE
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / f"{dataset}_run{run_index}_nn.joblib"
    joblib.dump(nn, model_path)

    record = build_experiment_record(
        dataset_name=dataset,
        dataset_version=DATASET_VERSION,
        split_method=SPLIT_METHOD,
        seed=SEED,
        preprocessing_version=PREPROCESSING_VERSION,
        model_type=MODEL_TYPE,
        fusion_strategy=FUSION_STRATEGY,
        hyperparameters=study.best_params,
        threshold=best_threshold,
        scores_test=scores_test,
        test_y=test_y,
        runtime_train=runtime_train,
        runtime_inference=runtime_inference,
        memory_peak=memory_peak,
        notes=run_cfg["notes"] + " | anomaly_score=distance_to_nearest_core_sample",
    )
    save_record_json(record, RESULTS_DIR, run_index, MODEL_TYPE, dataset)
    return record

In [ ]:
all_records = []

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)